## Using the Provided HOG and Colour Histogram CSV Files as Input Into K-Nearest Neighbours for Classification


This notebook contains code for a k-nearest neighbours algorithm which uses `hog_pca.csv` and `color_histogram.csv` as input. It also tests two different methodologies: scaling the `color_histogram.csv` data and not. Accuracy for non-scaled peaked at 33.73%, and accuracy for scaled peaked at 22.27%.


In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

train_metadata = pd.read_csv("./datasets/task1_data/train_metadata.csv")
hog_pca = pd.read_csv("./datasets/task1_data/hog_pca.csv")

hog_metadata_df = pd.merge(
    left=hog_pca, right=train_metadata, on="image_id", how="inner"
)


def calculate_distances(image, training_images):
    distances = euclidean_distances([image], training_images)
    return distances.flatten()


def do_knn(image, training_images, training_image_labels, k):
    distances = calculate_distances(image, training_images)
    sorted_distance_indices = np.argsort(distances)
    k_most_similar_indices = sorted_distance_indices[:k]
    labels = [training_image_labels.iloc[index] for index in k_most_similar_indices]
    label_vote_counts = Counter(labels)
    top_votes = label_vote_counts.most_common(2)
    if len(top_votes) > 1 and top_votes[0][1] == top_votes[1][1]:
        return do_knn(image, training_images, training_image_labels, k + 1)
    return top_votes[0][0]


def testing_pipeline(training_features, k, scale):
    train_df, val_df = train_test_split(
        training_features, test_size=0.2, random_state=2718
    )
    X_train = train_df.drop(
        columns=["image_id", "class_name", "class_id", "image_path"]
    )
    y_train = train_df["class_name"]

    X_val = val_df.drop(columns=["image_id", "class_name", "class_id", "image_path"])
    y_val = val_df["class_name"]

    if scale == "split":
        colour_cols = [col for col in X_train.columns if "color" in col]
        hog_cols = [col for col in X_train.columns if "hog" in col]

        X_train_colour = X_train[colour_cols]
        X_val_colour = X_val[colour_cols]

        scaler = StandardScaler()
        # the scaler fits to the mean and std of X_train
        X_train_colour_scaled = scaler.fit_transform(X=X_train_colour)
        # then remembers the mean and std and transforms X_val in accordance with that
        X_val_colour_scaled = scaler.transform(X_val_colour)

        # y_train and y_val aren't scaled because they're just class labels so they can't be scaled

        X_train_colour_scaled = pd.DataFrame(
            X_train_colour_scaled, columns=colour_cols, index=X_train.index
        )
        X_val_colour_scaled = pd.DataFrame(
            X_val_colour_scaled, columns=colour_cols, index=X_val.index
        )
        X_train_hog = X_train[hog_cols]
        X_val_hog = X_val[hog_cols]

        X_train = pd.concat([X_train_colour_scaled, X_train_hog], axis=1)
        X_val = pd.concat([X_val_colour_scaled, X_val_hog], axis=1)

    correct_preds = 0
    total_tests = len(X_val)

    print(f"Testing {total_tests} unseen images...")
    for i in range(total_tests):
        test_image_features = X_val.iloc[i]
        true_label = y_val.iloc[i]

        prediction = do_knn(
            image=test_image_features,
            training_images=X_train,
            training_image_labels=y_train,
            k=k,
        )

        if prediction == true_label:
            correct_preds += 1

    accuracy = correct_preds / total_tests
    print(f"Final accuracy: {accuracy:.4f}")
    return accuracy


def test_k_vals(df, lower_bound, upper_bound, scale):
    for k in range(lower_bound, upper_bound + 1):
        print(f"K-val: {k}")
        testing_pipeline(df, k, scale)


test_k_vals(df=hog_metadata_df, lower_bound=1, upper_bound=5, scale=False)

K-val: 1


ValueError: at least one array or dtype is required

In [11]:
colour_df = pd.read_csv("./datasets/task1_data/color_histogram.csv")
colour_hog_df = pd.merge(
    left=colour_df, right=hog_metadata_df, on="image_id", how="inner"
)

# print(f"total columns: {len(colour_hog_df.columns)}")
# print(colour_df.head())
print(hog_pca.columns)

Index(['image_id', 'hog_pca_0', 'hog_pca_1', 'hog_pca_2', 'hog_pca_3',
       'hog_pca_4', 'hog_pca_5', 'hog_pca_6', 'hog_pca_7', 'hog_pca_8',
       ...
       'hog_pca_90', 'hog_pca_91', 'hog_pca_92', 'hog_pca_93', 'hog_pca_94',
       'hog_pca_95', 'hog_pca_96', 'hog_pca_97', 'hog_pca_98', 'hog_pca_99'],
      dtype='object', length=101)


In [15]:
test_k_vals(colour_hog_df, 1, 15, scale="split")
print("-" * 60)

test_k_vals(colour_hog_df, 1, 15, scale=False)

# testing_pipeline(training_features=colour_hog_df, k=5)

K-val: 1
Testing 750 unseen images...
Final accuracy: 0.2000
K-val: 2
Testing 750 unseen images...
Final accuracy: 0.2200
K-val: 3
Testing 750 unseen images...
Final accuracy: 0.2200
K-val: 4
Testing 750 unseen images...
Final accuracy: 0.2213
K-val: 5
Testing 750 unseen images...
Final accuracy: 0.2173
K-val: 6
Testing 750 unseen images...
Final accuracy: 0.2107
K-val: 7
Testing 750 unseen images...
Final accuracy: 0.2080
K-val: 8
Testing 750 unseen images...
Final accuracy: 0.2160
K-val: 9
Testing 750 unseen images...
Final accuracy: 0.2173
K-val: 10
Testing 750 unseen images...
Final accuracy: 0.2160
K-val: 11
Testing 750 unseen images...
Final accuracy: 0.2147
K-val: 12
Testing 750 unseen images...
Final accuracy: 0.2133
K-val: 13
Testing 750 unseen images...
Final accuracy: 0.2173
K-val: 14
Testing 750 unseen images...
Final accuracy: 0.2227
K-val: 15
Testing 750 unseen images...
Final accuracy: 0.2227
------------------------------------------------------------
K-val: 1
Testing 7